<a href="https://colab.research.google.com/github/heetaamin/ml-assignment2/blob/main/final/Trees_Notebook1_Preprocessing_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 - Classification tree, global preprocessing

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/networkTraffic.csv'
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)

Mounted at /content/drive
Shape: (257673, 44)


In [2]:
# drop structurally redundant columns
# id is unique per row - keeping it just gives the tree room to overfit
# ct_ftp_cmd/sloss/dloss are near-duplicates of another retained feature and tcprtt is just synack + ackdat added together.

df_tree = df.copy()
df_tree = df_tree.drop(columns=['id', 'ct_ftp_cmd', 'sloss', 'dloss', 'tcprtt'])
print("Shape:", df_tree.shape)

Shape: (257673, 39)


In [3]:
# fix the two invalid-value issues from EDA

# state has one row = 'no', not in the documented categories - recode to the mode (FIN).

# is_ftp_login has 26 rows with 2 or 4 instead of 0/1. All 26 have a nonzero ct_ftp_cmd too, so a login must have happened - map to 1.

state_mode = df_tree['state'].mode()[0]
df_tree['state'] = df_tree['state'].replace('no', state_mode)

df_tree['is_ftp_login'] = df_tree['is_ftp_login'].replace({2: 1, 4: 1})

# confirm the fix with print statements
print("state fixed:", not (df_tree['state'] == 'no').any())
print("is_ftp_login fixed:", sorted(df_tree['is_ftp_login'].unique()))

state fixed: True
is_ftp_login fixed: [np.int64(0), np.int64(1)]


In [4]:
# check duplicates before any further processing
# checking now on raw values, so rows are not counted as duplicates because a later step collapsed them together

n_dupes = df_tree.duplicated().sum()
print(f"Duplicate rows: {n_dupes} ({n_dupes/len(df_tree)*100:.2f}%)")

Duplicate rows: 94928 (36.84%)


In [5]:
# drop the duplicates
# doing this before the CV split so a duplicated row can't end up in both train and test that would inflate accuracy without meaning
# Another benefit: duplication is concentrated in Generic and DoS, so this helps slightly with the class imbalance too.

print("Shape before dedup:", df_tree.shape)
df_tree = df_tree.drop_duplicates()
print("Shape after dedup:", df_tree.shape)

Shape before dedup: (257673, 39)
Shape after dedup: (162745, 39)


In [6]:
# save the semi-raw dataset
# service ('?') and proto (133 categories) untouched - those need to be fit inside the CV fold loop
# all 7 connection-count features retained - empirical test showed real signal in the dropped 5

df_tree.to_csv('/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/networkTraffic_tree_semiraw.csv', index=False)
print("Saved. Shape:", df_tree.shape)

Saved. Shape: (162745, 39)
